In [2]:
!nvidia-smi

Fri Mar  6 21:01:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!pwd

/content


In [46]:
%cd /content
!rm -rf ./Continual/

/content


In [47]:
!git clone https://github.com/nnminh322/Continual.git
!pip install uv
%cd Continual/gainlora_baseline_origin
# !bash setup_kaggle_colab.sh

Cloning into 'Continual'...
remote: Enumerating objects: 1550, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 1550 (delta 50), reused 47 (delta 24), pack-reused 1475 (from 1)
Receiving objects: 100% (1550/1550), 29.27 MiB | 21.55 MiB/s, done.
Resolving deltas: 100% (666/666), done.
Updating files: 100% (1597/1597), done.
/content/Continual/gainlora_baseline_origin


In [48]:
import subprocess, sys
# Use subprocess to avoid kernel's stale numpy/jax modules
result = subprocess.run([sys.executable, "-c", """
import torch, transformers, datasets, numpy, scipy, sys
print(f"python:       {sys.version.split()[0]}")
print(f"torch:        {torch.__version__}")
print(f"cuda:         {torch.cuda.is_available()}, devices={torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"gpu0:         {torch.cuda.get_device_name(0)}")
print(f"transformers: {transformers.__version__}")
print(f"datasets:     {datasets.__version__}")
print(f"numpy:        {numpy.__version__}")
print(f"scipy:        {scipy.__version__}")
try:
    import cupy; print(f"cupy:         {cupy.__version__}")
except: print("cupy:         NOT INSTALLED")
# Verify torch version is correct
assert torch.__version__.startswith("2.5.1"), f"WRONG TORCH: {torch.__version__}"
assert numpy.__version__ == "1.26.4", f"WRONG NUMPY: {numpy.__version__}"
print("\\nAll version checks PASSED")
"""], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Version check failed!")
import os; print(f"cwd: {os.getcwd()}")

python:       3.12.12
torch:        2.5.1+cu121
cuda:         True, devices=1
gpu0:         Tesla T4
transformers: 4.40.2
datasets:     2.21.0
numpy:        1.26.4
scipy:        1.14.1
cupy:         13.6.0

All version checks PASSED

cwd: /content/Continual/gainlora_baseline_origin


In [49]:
!bash gen_script_superni_order1_t5_specroute.sh 0 google-t5/t5-large

[GPU] Detected T4 GPUs (15360MB VRAM each)
[GPU] Strategy: 1x T4 + fp16
[GPU] Using CUDA_VISIBLE_DEVICES=0

2026-03-06 21:55:58.871171: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772834158.904013   18003 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772834158.914049   18003 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772834158.938527   18003 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772834158.938572   18003 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same ta

In [10]:
import subprocess
# Check for errors / tracebacks in the most recent training output
# Also check what files were produced
print("=== Checking for errors in output logs ===")
r = subprocess.run("find logs_and_outputs -name '*.out' -o -name '*.log' 2>/dev/null | head -5", shell=True, capture_output=True, text=True, cwd="/content/Continual/gainlora_baseline_origin")
print("Log files found:", r.stdout or "NONE")
print()
# Check if output directories were created
r2 = subprocess.run("ls -la logs_and_outputs/gen_script_superni_order1_t5_specroute/outputs/ 2>&1 | head -20", shell=True, capture_output=True, text=True, cwd="/content/Continual/gainlora_baseline_origin")
print("=== Output directories ===")
print(r2.stdout or r2.stderr)

=== Checking for errors in output logs ===
Log files found: NONE

=== Output directories ===
ls: cannot access 'logs_and_outputs/gen_script_superni_order1_t5_specroute/outputs/': No such file or directory



In [ ]:
import subprocess

# Re-run training but capture output to file for analysis
r = subprocess.run(
    "bash gen_script_superni_order1_t5_specroute.sh 0 google-t5/t5-large 2>&1 | tail -100",
    shell=True, capture_output=True, text=True, cwd="/content/Continual/gainlora_baseline_origin",
    timeout=600
)
print("STDOUT (last 100 lines):")
print(r.stdout[-3000:] if r.stdout else "EMPTY")
print("STDERR:")
print(r.stderr[-1000:] if r.stderr else "EMPTY")
print("Return code:", r.returncode)

=== Full tree of output ===
logs_and_outputs/gen_script_superni_order1_t5_specroute/outputs/1-task1572_samsum_summary/runs/Mar06_21-56-04_785b77d0687e/events.out.tfevents.1772834173.785b77d0687e.18003.0

=== Check for .out files (stdout logs) ===
No .out files
=== Check for .err files ===
No .err files
